# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [1]:
# Load the libraries as required.
%load_ext dotenv
%dotenv 
import os
import sys
sys.path.append(os.getenv('SRC_DIR'))
import pandas as pd
import numpy as np
import os

In [2]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [3]:
X = fires_dt.drop(columns=['area']) #Features df
y = fires_dt['area'] #target

In [4]:
X.sample(5)

,coord_x,coord_y,month,day,ffmc,dmc,dc,isi,temp,rh,wind,rain
231,1,5,sep,sun,93.5,149.3,728.6,8.1,27.8,27,3.1,0.0
64,2,2,aug,mon,91.1,103.2,638.8,5.8,23.1,31,3.1,0.0
200,1,5,sep,tue,91.0,129.5,692.6,7.0,21.6,33,2.2,0.0
120,3,4,aug,mon,91.5,145.4,608.2,10.7,10.3,74,2.2,0.0
255,2,5,aug,thu,87.5,77.0,694.8,5.0,22.3,46,4.0,0.0


In [5]:
y.sample(5)

140     0.47
343     2.18
55      0.00
375    39.35
309     0.00
Name: area, dtype: float64

In [6]:
X.shape, y.shape

((517, 12), (517,))

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, PowerTransformer, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor

In [8]:
#Here I want to know which columns are numerical and which are categorical
num_features = X.select_dtypes(include=['float64', 'int64']).columns
cat_features = X.select_dtypes(include=['object']).columns
num_features, cat_features

(Index(['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind',
        'rain'],
       dtype='object'),
 Index(['month', 'day'], dtype='object'))

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) #Splitting the data into train and test sets

In [10]:
#Preprocessor 1
preproc1 = ColumnTransformer([
    ('Scaler', StandardScaler(), num_features),
    ("OHE", OneHotEncoder(handle_unknown='ignore'), cat_features)],
    remainder='passthrough'
)

### Preproc 2

Create preproc2 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [11]:
preproc2 = ColumnTransformer([
    ('Q_Scaler', PowerTransformer(method='yeo-johnson'), num_features),
    ("OHE", OneHotEncoder(handle_unknown='ignore'), cat_features)],
    remainder='passthrough')

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [12]:
# Pipeline A = preproc1 + baseline

pipe_a = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', KNeighborsRegressor())
])

In [13]:
# Pipeline B = preproc2 + baseline
pipe_b = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', KNeighborsRegressor())
])

In [14]:
# Pipeline C = preproc1 + advanced model
pipe_c = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', GradientBoostingRegressor())
])


In [15]:
# Pipeline D = preproc2 + advanced model
pipe_d = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', GradientBoostingRegressor())
])
    

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [16]:
from sklearn.model_selection import GridSearchCV

In [17]:
param_grid_KNN = {
    'regressor__n_neighbors': [3,5,7,9]
}

param_grid_boosting = {
    'regressor__n_estimators': [50,100,200,300],
    'regressor__learning_rate': [0.01,0.05, 0.1, 0.2]
}

scoring_metrics = [
    'neg_root_mean_squared_error',  # RMSE (negative for minimization)
    'neg_mean_absolute_error',     # MAE (negative for minimization)
    'neg_mean_squared_error',      # MSE (negative for minimization)
    'r2',                          # R-squared (higher is better)
]


In [21]:
grid_search_a = GridSearchCV(
    pipe_a, 
    param_grid_KNN, 
    cv=5,
    scoring=scoring_metrics,
    refit='neg_mean_squared_error')

grid_search_a.fit(X_train, y_train)

results_a = pd.DataFrame(grid_search_a.cv_results_)
results_a.sort_values('rank_test_neg_mean_squared_error').head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_regressor__n_neighbors,params,split0_test_neg_root_mean_squared_error,split1_test_neg_root_mean_squared_error,split2_test_neg_root_mean_squared_error,split3_test_neg_root_mean_squared_error,...,std_test_neg_mean_squared_error,rank_test_neg_mean_squared_error,split0_test_r2,split1_test_r2,split2_test_r2,split3_test_r2,split4_test_r2,mean_test_r2,std_test_r2,rank_test_r2
3,0.004661,0.006240,0.011004,0.009463,9,{'regressor__n_neighbors': 9},-41.587015,-25.319218,-29.744026,-85.778561,...,2582.757809,1,-0.060177,-1.963334,-0.228203,-0.021260,-0.118672,-0.478329,0.745777,1
2,0.007198,0.004264,0.004527,0.003889,7,{'regressor__n_neighbors': 7},-43.706496,-27.800463,-31.378079,-86.031554,...,2577.737077,2,-0.170995,-2.572597,-0.366857,-0.027293,0.027045,-0.622139,0.984710,2
1,0.000772,0.001544,0.012200,0.006229,5,{'regressor__n_neighbors': 5},-44.646732,-35.932717,-34.225537,-87.689556,...,2601.359991,3,-0.221919,-4.968430,-0.626189,-0.067271,-0.099719,-1.196706,1.896344,3
0,0.017089,0.007641,0.015270,0.010113,3,{'regressor__n_neighbors': 3},-53.040798,-40.102857,-45.851923,-88.388739,...,2440.098164,4,-0.724579,-6.434138,-1.918674,-0.084358,-0.763544,-1.985059,2.301838,4


In [23]:
print(f"Best score: {grid_search_a.best_score_}")
print(f"Best params: {grid_search_a.best_params_}")

Best score: -2253.9980358272114
Best params: {'regressor__n_neighbors': 9}


In [24]:
grid_search_b = GridSearchCV(
    pipe_b, 
    param_grid_KNN, 
    cv=5,
    scoring=scoring_metrics,
    refit='neg_mean_squared_error')

grid_search_b.fit(X_train, y_train)

results_b = pd.DataFrame(grid_search_b.cv_results_)
results_b.sort_values('rank_test_neg_mean_squared_error').head()


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_regressor__n_neighbors,params,split0_test_neg_root_mean_squared_error,split1_test_neg_root_mean_squared_error,split2_test_neg_root_mean_squared_error,split3_test_neg_root_mean_squared_error,...,std_test_neg_mean_squared_error,rank_test_neg_mean_squared_error,split0_test_r2,split1_test_r2,split2_test_r2,split3_test_r2,split4_test_r2,mean_test_r2,std_test_r2,rank_test_r2
3,0.015481,0.001399,0.008517,0.000744,9,{'regressor__n_neighbors': 9},-41.982912,-25.910507,-29.373916,-84.885766,...,2516.332041,1,-0.080458,-2.103358,-0.197827,-0.000112,-0.167111,-0.509773,0.799788,1
2,0.015385,0.002451,0.011469,0.003252,7,{'regressor__n_neighbors': 7},-42.586015,-28.500154,-30.654128,-85.954218,...,2552.607665,2,-0.111724,-2.754693,-0.304513,-0.025447,-0.279966,-0.695269,1.034947,2
1,0.018749,0.003220,0.008324,0.000396,5,{'regressor__n_neighbors': 5},-44.863294,-36.107718,-31.055147,-86.920519,...,2532.152193,3,-0.233801,-5.026707,-0.338868,-0.048633,-0.569903,-1.243582,1.899050,3
0,0.021311,0.010318,0.009977,0.004998,3,{'regressor__n_neighbors': 3},-52.750456,-48.640640,-37.547045,-89.067861,...,2491.951313,4,-0.705750,-9.936505,-0.957141,-0.101085,-0.791834,-2.498463,3.730254,4


In [25]:
print(f"Best score: {grid_search_b.best_score_}")
print(f"Best params: {grid_search_b.best_params_}")

Best score: -2237.5113382088425
Best params: {'regressor__n_neighbors': 9}


In [26]:
grid_search_c = GridSearchCV(
    pipe_c, 
    param_grid_boosting, 
    cv=5,
    scoring=scoring_metrics,
    refit='neg_mean_squared_error')

grid_search_c.fit(X_train, y_train)

results_c = pd.DataFrame(grid_search_c.cv_results_)
results_c.sort_values('rank_test_neg_mean_squared_error').head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_regressor__learning_rate,param_regressor__n_estimators,params,split0_test_neg_root_mean_squared_error,split1_test_neg_root_mean_squared_error,split2_test_neg_root_mean_squared_error,...,std_test_neg_mean_squared_error,rank_test_neg_mean_squared_error,split0_test_r2,split1_test_r2,split2_test_r2,split3_test_r2,split4_test_r2,mean_test_r2,std_test_r2,rank_test_r2
0,0.080615,0.026874,0.004629,0.003944,0.01,50,"{'regressor__learning_rate': 0.01, 'regressor_...",-49.516197,-39.874690,-26.387911,...,2462.252520,1,-0.502995,-6.349785,0.033325,-0.011884,-0.168276,-1.399923,2.482073,1
1,0.126083,0.009245,0.004244,0.002802,0.01,100,"{'regressor__learning_rate': 0.01, 'regressor_...",-55.355668,-60.427577,-36.650101,...,2330.702260,2,-0.878396,-15.879142,-0.864751,-0.014228,-0.067688,-3.540841,6.180348,2
2,0.229610,0.013285,0.005178,0.003834,0.01,200,"{'regressor__learning_rate': 0.01, 'regressor_...",-63.216602,-71.598464,-45.330192,...,2320.300943,3,-1.449770,-22.696675,-1.852631,-0.023174,-0.268256,-5.258101,8.746514,3
3,0.404194,0.046796,0.006919,0.001801,0.01,300,"{'regressor__learning_rate': 0.01, 'regressor_...",-64.316599,-72.533051,-46.110901,...,2316.692754,4,-1.535765,-23.319347,-1.951737,-0.030695,-0.430323,-5.453573,8.960341,5
4,0.068885,0.011206,0.005028,0.002600,0.05,50,"{'regressor__learning_rate': 0.05, 'regressor_...",-67.710942,-70.994693,-51.315199,...,2282.216817,5,-1.810481,-22.298705,-2.655633,-0.032144,-0.171411,-5.393675,8.510397,4


In [27]:
print(f"Best score: {grid_search_c.best_score_}")
print(f"Best params: {grid_search_c.best_params_}")

Best score: -2542.894986879389
Best params: {'regressor__learning_rate': 0.01, 'regressor__n_estimators': 50}


In [28]:
grid_search_d = GridSearchCV(
    pipe_d, 
    param_grid_boosting, 
    cv=5,
    scoring=scoring_metrics,
    refit='neg_mean_squared_error')

grid_search_d.fit(X_train, y_train)

results_d = pd.DataFrame(grid_search_d.cv_results_)
results_d.sort_values('rank_test_neg_mean_squared_error').head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_regressor__learning_rate,param_regressor__n_estimators,params,split0_test_neg_root_mean_squared_error,split1_test_neg_root_mean_squared_error,split2_test_neg_root_mean_squared_error,...,std_test_neg_mean_squared_error,rank_test_neg_mean_squared_error,split0_test_r2,split1_test_r2,split2_test_r2,split3_test_r2,split4_test_r2,mean_test_r2,std_test_r2,rank_test_r2
0,0.074599,0.010332,0.007332,0.005110,0.01,50,"{'regressor__learning_rate': 0.01, 'regressor_...",-48.899098,-40.503156,-26.128838,...,2463.030518,1,-0.465766,-6.583291,0.052213,-0.011626,-0.138278,-1.429350,2.583155,1
1,0.137149,0.009021,0.004566,0.003782,0.01,100,"{'regressor__learning_rate': 0.01, 'regressor_...",-54.683662,-61.975223,-35.410633,...,2352.733526,2,-0.833066,-16.754817,-0.740756,-0.015756,-0.116323,-3.692144,6.539431,2
2,0.282996,0.036665,0.017688,0.024089,0.01,200,"{'regressor__learning_rate': 0.01, 'regressor_...",-62.166349,-71.214269,-45.053136,...,2302.892691,3,-1.369047,-22.443047,-1.817867,-0.024336,-0.376953,-5.206250,8.642763,3
3,0.374858,0.019683,0.005816,0.005415,0.01,300,"{'regressor__learning_rate': 0.01, 'regressor_...",-61.691306,-71.566031,-46.095145,...,2309.197585,4,-1.332979,-22.675212,-1.949720,-0.031586,-0.381926,-5.274285,8.726969,4
4,0.085527,0.006251,0.002569,0.003210,0.05,50,"{'regressor__learning_rate': 0.05, 'regressor_...",-64.813151,-71.643402,-50.295584,...,2284.491235,5,-1.575071,-22.726431,-2.511803,-0.032201,-0.216180,-5.412337,8.704728,5


In [29]:
print(f"Best score: {grid_search_d.best_score_}")
print(f"Best params: {grid_search_d.best_params_}")

Best score: -2534.2361826982096
Best params: {'regressor__learning_rate': 0.01, 'regressor__n_estimators': 50}


# Evaluate

+ Which model has the best performance?

In [36]:
# Collecting results for each model
results = {
    "Model A": {
        "Best Score (NMSE)": grid_search_a.best_score_,
        "Best Params": grid_search_a.best_params_
    },
    "Model B": {
        "Best Score (NMSE)": grid_search_b.best_score_,
        "Best Params": grid_search_b.best_params_
    },
    "Model C": {
        "Best Score (NMSE)": grid_search_c.best_score_,
        "Best Params": grid_search_c.best_params_
    },
    "Model D": {
        "Best Score (NMSE)": grid_search_d.best_score_,
        "Best Params": grid_search_d.best_params_
    }
}

# Display the results
import pandas as pd
comparison_df = pd.DataFrame(results).T
print(comparison_df)


        Best Score (NMSE)                                        Best Params
Model A      -2253.998036                      {'regressor__n_neighbors': 9}
Model B      -2237.511338                      {'regressor__n_neighbors': 9}
Model C      -2542.894987  {'regressor__learning_rate': 0.01, 'regressor_...
Model D      -2534.236183  {'regressor__learning_rate': 0.01, 'regressor_...


The best model is Model B, because it has the lowest MSE.

# Export

+ Save the best performing model to a pickle file.

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.